# Document Preprocessing for Dementia Simulation

This notebook processes uploaded PDFs and CSV files:
1. Cleans and extracts text from PDFs
2. Processes CSV data
3. Chunks text into passages
4. Saves processed data to `data/processed/`

In [ ]:
import os
import pandas as pd
import PyPDF2
import nltk
import json
import re
from pathlib import Path
from typing import List, Dict, Any
from tqdm import tqdm

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

In [ ]:
# Configuration
UPLOAD_DIR = "data/uploads/"  # Directory containing uploaded files
PROCESSED_DIR = "data/processed/"
CHUNK_SIZE = 512  # Maximum tokens per chunk
OVERLAP = 50      # Overlap between chunks

# Ensure directories exist
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Upload directory: {UPLOAD_DIR}")
print(f"Processed directory: {PROCESSED_DIR}")

In [ ]:
def clean_text(text: str) -> str:
    """Clean and normalize text content."""
    # Remove extra whitespace and normalize line breaks
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\n+', '\n', text)
    
    # Remove special characters but keep punctuation
    text = re.sub(r'[^\w\s.,!?;:()\"\'-]', '', text)
    
    # Strip leading/trailing whitespace
    return text.strip()

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract text content from PDF file."""
    text = ""
    
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            
            for page_num in range(len(pdf_reader.pages)):
                page = pdf_reader.pages[page_num]
                page_text = page.extract_text()
                text += page_text + "\n"
                
    except Exception as e:
        print(f"Error reading PDF {pdf_path}: {str(e)}")
        return ""
    
    return clean_text(text)

def process_csv_file(csv_path: str) -> str:
    """Process CSV file and extract text content."""
    try:
        df = pd.read_csv(csv_path)
        
        # Combine all text columns into a single text string
        text_content = ""
        
        for column in df.columns:
            if df[column].dtype == 'object':  # Text columns
                column_text = df[column].fillna('').astype(str)
                text_content += f"\n{column}: \n" + "\n".join(column_text.tolist()) + "\n"
        
        return clean_text(text_content)
        
    except Exception as e:
        print(f"Error reading CSV {csv_path}: {str(e)}")
        return ""

In [ ]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = OVERLAP) -> List[str]:
    """Split text into overlapping chunks."""
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = ""
    current_length = 0
    
    for sentence in sentences:
        sentence_length = len(word_tokenize(sentence))
        
        if current_length + sentence_length > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            
            # Create overlap by keeping the last few sentences
            overlap_sentences = sent_tokenize(current_chunk)
            overlap_text = " ".join(overlap_sentences[-2:]) if len(overlap_sentences) > 1 else ""
            overlap_length = len(word_tokenize(overlap_text))
            
            if overlap_length <= overlap:
                current_chunk = overlap_text + " " + sentence
                current_length = overlap_length + sentence_length
            else:
                current_chunk = sentence
                current_length = sentence_length
        else:
            current_chunk += " " + sentence if current_chunk else sentence
            current_length += sentence_length
    
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    
    return chunks

def process_document(file_path: str) -> Dict[str, Any]:
    """Process a single document (PDF or CSV) and return metadata and chunks."""
    file_extension = Path(file_path).suffix.lower()
    file_name = Path(file_path).name
    
    if file_extension == '.pdf':
        text = extract_text_from_pdf(file_path)
        file_type = 'pdf'
    elif file_extension == '.csv':
        text = process_csv_file(file_path)
        file_type = 'csv'
    else:
        print(f"Unsupported file type: {file_extension}")
        return None
    
    if not text.strip():
        print(f"No text extracted from {file_name}")
        return None
    
    chunks = chunk_text(text)
    
    return {
        'file_name': file_name,
        'file_type': file_type,
        'file_path': file_path,
        'text_length': len(text),
        'num_chunks': len(chunks),
        'chunks': chunks
    }

In [ ]:
# Process all uploaded files
def process_all_documents():
    """Process all documents in the upload directory."""
    processed_documents = []
    all_chunks = []
    
    # Find all PDF and CSV files in upload directory
    upload_path = Path(UPLOAD_DIR)
    pdf_files = list(upload_path.glob("*.pdf"))
    csv_files = list(upload_path.glob("*.csv"))
    
    all_files = pdf_files + csv_files
    
    if not all_files:
        print(f"No PDF or CSV files found in {UPLOAD_DIR}")
        print("Creating sample data for demonstration...")
        create_sample_data()
        return process_all_documents()  # Retry after creating sample data
    
    print(f"Found {len(all_files)} files to process: {len(pdf_files)} PDFs, {len(csv_files)} CSVs")
    
    for file_path in tqdm(all_files, desc="Processing documents"):
        doc_data = process_document(str(file_path))
        
        if doc_data:
            processed_documents.append(doc_data)
            
            # Add chunks with metadata
            for i, chunk in enumerate(doc_data['chunks']):
                chunk_data = {
                    'id': f"{doc_data['file_name']}_chunk_{i}",
                    'text': chunk,
                    'source_file': doc_data['file_name'],
                    'file_type': doc_data['file_type'],
                    'chunk_index': i
                }
                all_chunks.append(chunk_data)
    
    return processed_documents, all_chunks

def create_sample_data():
    """Create sample data for demonstration purposes."""
    # Create sample CSV
    sample_csv_data = {
        'Patient_ID': ['P001', 'P002', 'P003', 'P004', 'P005'],
        'Age': [75, 68, 82, 71, 79],
        'Diagnosis': ['Mild Cognitive Impairment', 'Early-stage Alzheimer\'s', 
                     'Moderate Dementia', 'Vascular Dementia', 'Frontotemporal Dementia'],
        'Symptoms': [
            'Memory lapses, difficulty with complex tasks',
            'Short-term memory loss, confusion with familiar places',
            'Severe memory problems, behavioral changes, difficulty recognizing family',
            'Problems with planning and organizing, mood changes',
            'Personality changes, language difficulties, impaired judgment'
        ],
        'Care_Notes': [
            'Patient benefits from routine and familiar environments',
            'Requires assistance with medication management',
            'Needs constant supervision and assistance with daily activities',
            'Responds well to structured activities and social interaction',
            'Benefits from calm environment and clear, simple communication'
        ]
    }
    
    sample_df = pd.DataFrame(sample_csv_data)
    sample_df.to_csv(os.path.join(UPLOAD_DIR, 'sample_patient_data.csv'), index=False)
    
    # Create sample text file (simulating PDF content)
    sample_text = """
Dementia Care Guidelines

Introduction
Dementia is a syndrome characterized by deterioration in memory, thinking, behavior and the ability to perform everyday activities. While dementia mainly affects older people, it is not a normal part of ageing.

Types of Dementia
Alzheimer's disease is the most common form of dementia and may contribute to 60–70% of cases. Other major forms include vascular dementia, dementia with Lewy bodies, and frontotemporal dementia.

Symptoms and Stages
Early stage symptoms include forgetfulness, losing track of time, and becoming lost in familiar places. Middle stage symptoms involve forgetting recent events and people's names, becoming confused at home, and having difficulty with communication. Late stage symptoms include becoming unaware of time and place, having difficulty recognizing relatives and friends, and needing assisted self-care.

Care Strategies
Effective care strategies include maintaining routine, providing clear and simple instructions, ensuring a safe environment, and promoting social interaction. Family involvement and professional support are crucial for quality care.

Communication Tips
When communicating with dementia patients, speak slowly and clearly, use simple words, maintain eye contact, and be patient. Avoid arguing or correcting, and instead redirect attention to positive activities.

Environmental Considerations
The environment should be safe, familiar, and calming. Remove potential hazards, ensure good lighting, minimize noise, and provide clear pathways. Personal items and photos can help maintain connection to identity.
"""
    
    with open(os.path.join(UPLOAD_DIR, 'dementia_care_guide.txt'), 'w') as f:
        f.write(sample_text)
    
    print("Sample data created in upload directory")

In [ ]:
# Run the processing pipeline
processed_docs, chunks = process_all_documents()

print(f"\nProcessing Summary:")
print(f"- Processed {len(processed_docs)} documents")
print(f"- Generated {len(chunks)} text chunks")

if processed_docs:
    print("\nDocument Details:")
    for doc in processed_docs:
        print(f"- {doc['file_name']} ({doc['file_type']}): {doc['num_chunks']} chunks, {doc['text_length']} characters")

In [ ]:
# Save processed data
def save_processed_data(processed_docs, chunks):
    """Save processed documents and chunks to files."""
    
    # Save document metadata
    metadata_file = os.path.join(PROCESSED_DIR, 'document_metadata.json')
    with open(metadata_file, 'w') as f:
        json.dump(processed_docs, f, indent=2)
    
    # Save chunks
    chunks_file = os.path.join(PROCESSED_DIR, 'text_chunks.json')
    with open(chunks_file, 'w') as f:
        json.dump(chunks, f, indent=2)
    
    # Save chunks as CSV for easy viewing
    chunks_df = pd.DataFrame(chunks)
    chunks_csv = os.path.join(PROCESSED_DIR, 'text_chunks.csv')
    chunks_df.to_csv(chunks_csv, index=False)
    
    print(f"\nSaved processed data:")
    print(f"- Document metadata: {metadata_file}")
    print(f"- Text chunks (JSON): {chunks_file}")
    print(f"- Text chunks (CSV): {chunks_csv}")
    
    return chunks_file

# Save the processed data
chunks_file = save_processed_data(processed_docs, chunks)

In [ ]:
# Preview some chunks
print("\nSample chunks:")
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1} (ID: {chunk['id']}):")
    print(f"Source: {chunk['source_file']}")
    print(f"Text: {chunk['text'][:200]}...")
    print("-" * 80)

## Next Steps

The preprocessed text chunks are now ready for embedding and indexing. Run the `build_index.py` script to:

1. Load the processed chunks
2. Generate embeddings using sentence transformers
3. Create and save a FAISS index

```bash
python build_index.py
```